# Mimic Jev (The Dep) — the whole trick in 3 steps

0. **Parse the JSON spec**: `{state, questions}` → one prompt per question.
1. **Reprompt**: add marks (`1,2,3`) to the options + "Reply with exactly one character."
2. **`logit_bias`**: +50 on every candidate token so nothing else can win.
3. **Renormalise** the returned probs over the candidates only — equal bias cancels, so relative belief is untouched.

That's `minijev.decide()`. The rest of this notebook is demos + one warning box.

## Step 0 — parse the sample JSON

The request arrives as JSON: `{state, questions: {name: {type, instructions, criteria}}}`. Parse it, dispatch on `type`, build one prompt per question. (`criteria` shape differs: noul = optional `{true, false}` dict, choice = `{option: description}` dict, score = ordered list.)

In [ ]:
import json

ASK = "<|im_start|>user\n{body}<|im_end|>\n<|im_start|>assistant\n<think></think>Answer:"

def reprompt(state, question, options):
    marks = [str(i) for i in range(1, len(options) + 1)]
    menu = "\n".join(m + ". " + k + " -- " + v for m, k, v in zip(marks, options, options.values()))
    body = "Text:\n" + state + "\n\nQuestion: " + question + "\nOptions:\n" + menu + "\nReply with exactly one character."
    return ASK.format(body=body), {k: " " + m for k, m in zip(options, marks)}

SPEC = '{"state": "limits reset for the day.", "questions": {"is_quota_reset": {"type": "noul", "instructions": "Does this announce a quota reset?"}, "urgency": {"type": "choice", "instructions": "How urgent is this?", "criteria": {"ignore": "not relevant", "today": "act today", "now": "stop"}}}}'

def build_prompt(state, q):
    t = q["type"]
    if t == "noul":
        crit = q.get("criteria")
        extra = "" if not crit else "\ntrue means: " + crit["true"] + "\nfalse means: " + crit["false"]
        body = "Text:\n" + state + "\n\nQuestion: " + q["instructions"] + extra + "\nReply with exactly one word: Yes or No."
        return ASK.format(body=body), {"true": " Yes", "false": " No"}
    if t == "choice":
        return reprompt(state, q["instructions"], q["criteria"])
    marks = [str(i) for i in range(1, len(q["criteria"]) + 1)]
    menu = "\n".join(m + ". " + d for m, d in zip(marks, q["criteria"]))
    body = "Text:\n" + state + "\n\nQuestion: " + q["instructions"] + "\nScale:\n" + menu + "\nReply with exactly one character."
    return ASK.format(body=body), {str(i + 1): " " + m for i, m in enumerate(marks)}

req = json.loads(SPEC)
for name, q in req["questions"].items():
    prompt, labels = build_prompt(req["state"], q)
    print("---", name, q["type"], "->", labels)


## Step 1 — reprompt: marks + one-token constraint

Take any options JSON and render it as a numbered menu. The prompt does almost nothing — the enforcement happens in steps 2–3.

In [ ]:
prompt, labels = reprompt("limits reset for the day.", "How urgent is this?", {"ignore": "not relevant", "today": "act today", "now": "stop what you are doing"})
print(prompt)
print(labels)


## Steps 2+3 — equal bias forces the schema, renormalise keeps belief

Send `logit_bias=[[id, 50], ...]` (equal on every candidate) with `n_predict=1`. Then softmax over the candidates only.

Why the numbers stay honest — the bias cancels:

$$ \frac{e^{x_i+b}}{\sum_j e^{x_j+b}} = \frac{e^{x_i}}{\sum_j e^{x_j}} $$

A value outside your schema isn't unlikely — it's **unrepresentable**. Prompt injection can't produce a token that was masked out.

In [ ]:
import math

def softmax(xs):
    m = max(xs)
    es = [math.exp(x - m) for x in xs]
    return [e / sum(es) for e in es]

class FakeLlama:
    logit = {" 1": -1.0, " 2": 1.5, " 3": 0.8, " Maybe": 5.0}
    def completion(self, labels, bias=50.0):
        boosted = {t: self.logit.get(t, -5.0) + (bias if t in labels.values() else 0.0) for t in self.logit}
        full = softmax(list(boosted.values()))
        keys = list(boosted)
        raw = {n: full[keys.index(t)] for n, t in labels.items()}
        z = sum(raw.values())
        return {k: v / z for k, v in raw.items()}

p = FakeLlama().completion({"ignore": " 1", "today": " 2", "now": " 3"})
print({k: round(v, 4) for k, v in p.items()})
print("pick:", max(p, key=p.get))
top2 = sorted(p.values(), reverse=True)[:2]
print("confidence (top minus runner-up):", round(top2[0] - top2[1], 4))


## Warning — the only two ways this silently breaks

1. **`temperature` must be 1.0.** At 0 the distribution collapses to one-hot — your pick survives but the calibrated probs (the whole point) are gone.
2. **`post_sampling_probs: True`.** The default report shows PRE-bias logits, ranked by the unbiased distribution — a disliked candidate falls out of the reporting window and reads as a silent zero. If any candidate is missing from the report, **raise instead of scoring it 0**.

Same one token gives you all three primitives for free: `noul` = Yes/No labels, `choice` = argmax over marks, `score` = expectation over ordered ranks (turns N levels into one real number, no regression head).